# CodeGen — Group 45

## Step 6 — RAG sweep on Qwen2.5-Coder-1.5B (+ compile-guided cascade)

**Where we are.** The vanilla Qwen2.5-Coder-1.5B baseline (Step 5) scores **37.8%**
(59/156) on MultiPL-E `humaneval-rs`. Error analysis of the 97 failures (2026-07-17)
found:

- **34 compile errors**, of which **20 fail with a single rustc error** — mostly
  type-discipline slips (`isize` used to index a slice, `f64` has no `Ord`,
  int/float mixing) plus **8 "phantom helpers"**: the model calls `is_prime` /
  `factorial` it never defined — an artifact of the single-function completion
  format (a nested `fn` would have been legal).
- **~61 logic errors** (wrong-answer asserts, index-out-of-bounds, overflow) that no
  prompt trick is likely to fix.
- `162_string_to_md5` needs an external crate → unwinnable in single-file rustc;
  the real ceiling is 155/156.

**What we test here** — the same experiment we ran for codegen-350M in Step 4, so
the two sweeps are directly comparable across model scales:

1. **RAG sweep K = 0/1/2/4** — retrieve MBPP Rust exemplars (TF-IDF) and prepend
   them to the completion prompt. At 350M, RAG *hurt* (7.1% → 5.8 / 4.5 / 5.8); we
   attributed that to a fine-tune that had never seen in-context examples and to
   350M-scale weakness at ICL. Qwen-1.5B is a real ICL model, so the result may
   flip. **K=0 is the control and must reproduce 37.8% exactly.**
2. **An error-analysis-guided corpus row** — a handful of exemplars demonstrating
   exactly the idioms the compile errors need (`as usize` casts, `partial_cmp` on
   floats, nested helper `fn`s).
3. **The compile-guided cascade** (K=0 → on compile failure, fall back to K>0) —
   pure post-processing of the sweep files, zero extra GPU time. At 350M this was
   our best number (10.3%); here it has a hard target: the baseline's 34 compile
   errors.

| Reference rows (same harness) | Score |
|---|---|
| codegen-350M vanilla | 1.3% |
| codegen-350M translation fine-tune (Step 3b) | 7.1% |
| codegen-350M compile-guided cascade (Step 4) | 10.3% |
| **Qwen2.5-Coder-1.5B vanilla (Step 5)** | **37.8%** |

House rules apply: smoke test before every GPU run, every run streams to Drive and
resumes, nothing downloads twice.

## 0. Colab setup — Drive + Hugging Face token (run this first)

Everything we produce (benchmark file, model copy, eval results) lives in Drive at
`MyDrive/CodeGen_Group45`, so a crashed or recycled Colab session never loses work.

**One-time setup:** add a Colab secret (key icon in the left sidebar) named `HF_TOKEN`
containing a Hugging Face **read** token, and switch **Notebook access** ON for it.
Unauthenticated downloads from Colab are exactly what stalls / 403s (July 2026).

In [1]:
import os

IN_COLAB = "COLAB_RELEASE_TAG" in os.environ
DRIVE_ROOT = None

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_ROOT = "/content/drive/MyDrive/CodeGen_Group45"
    for sub in ("data", "models", "eval"):
        os.makedirs(os.path.join(DRIVE_ROOT, sub), exist_ok=True)

    # HF auth BEFORE anything talks to the Hub. Colab secret: HF_TOKEN, Notebook access ON.
    try:
        from google.colab import userdata
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
        print("HF token loaded from Colab secret")
    except Exception as e:
        print(f"WARNING: could not read the HF_TOKEN secret ({type(e).__name__}). "
              "Hub downloads may stall or 403 — add the secret and enable Notebook access.")
else:
    print("Not on Colab — skipping Drive; the benchmark loads from the repo's data/ folder.")

# Escape hatch only — leave False. With an upgraded hf_xet + auth, the Xet backend is the
# path that works from Colab; the non-Xet fallback was 403ing server-side (July 2026).
DISABLE_XET = False
if DISABLE_XET:
    os.environ["HF_HUB_DISABLE_XET"] = "1"

print("DRIVE_ROOT =", DRIVE_ROOT)

Mounted at /content/drive
HF token loaded from Colab secret
DRIVE_ROOT = /content/drive/MyDrive/CodeGen_Group45


## 1. Install the Rust toolchain
This gives us `rustc` (the Rust compiler). Takes ~1 minute.

In [2]:
# Install Rust (non-interactive)
!curl https://sh.rustup.rs -sSf | sh -s -- -y -q

# Make rustc/cargo visible to this notebook
import os
os.environ["PATH"] = os.path.expanduser("~/.cargo/bin") + ":" + os.environ["PATH"]

# Verify
!rustc --version
!cargo --version

warn: It looks like you have an existing rustup settings file at:
warn: /root/.rustup/settings.toml
warn: Rustup will install the default toolchain as specified in the settings file,
warn: instead of the one inferred from the default host triple.

  stable-x86_64-unknown-linux-gnu installed - rustc 1.97.1 (8bab26f4f 2026-07-14)


Rust is installed now. Great!

To get started you may need to restart your current shell.
This would reload your PATH environment variable to include
Cargo's bin directory ($HOME/.cargo/bin).

To configure your current shell, you need to source
the corresponding env file under $HOME/.cargo.

This is usually done by running one of the following (note the leading DOT):
. "$HOME/.cargo/env"            # For sh/bash/zsh/ash/dash/pdksh
source "$HOME/.cargo/env.fish"  # For fish
source "~/.cargo/env.nu"  # For nushell
source "$HOME/.cargo/env.tcsh"  # For tcsh
. "$HOME/.cargo/env.ps1"        # For pwsh
source "$HOME/.cargo/env.xsh"   # For xonsh
rustc 1.97.1 (8bab26

## 2. Install Python dependencies

Only `huggingface_hub` + its `hf_xet` download backend — and we **upgrade** them, because
Colab's preinstalled `hf_xet` is exactly what stalled our model downloads.

**Deliberately NOT installed: `datasets`.** `pip install -U datasets` drags a newer pyarrow
over Colab's preinstalled one and crashes the runtime (`IpcReadOptions size changed`).
This notebook never imports `datasets` at all — the benchmark is a plain JSONL (Section 3).

In [3]:
# Upgrade the Hub client + Xet backend BEFORE anything imports huggingface_hub.
# Do NOT add `datasets` or `torch` here (see the markdown above).
!pip install -q -U huggingface_hub hf_xet
print("done")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 771.9/771.9 kB 43.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.4/4.4 MB 13.9 MB/s eta 0:00:00
done


## 3. Load the MultiPL-E Rust problems
`humaneval-rs` = 156 classic coding problems, translated into Rust, **with unit tests**.
Each problem has:
- **prompt** — the function signature + a doc comment (ends with an open `{`)
- **tests** — a `fn main()` full of `assert_eq!` checks (starts with the closing `}`)

So a complete program is simply: **prompt + the model's body + tests**.

We keep the benchmark as a plain JSONL file (repo: `data/humaneval_rs.jsonl`, Drive:
`CodeGen_Group45/data/humaneval_rs.jsonl`) and read it with stdlib `json` — no `datasets`
library, no Hub download, nothing to flake. `ds` is a plain list of dicts.

In [4]:
import json, os

def load_benchmark():
    candidates = []
    if DRIVE_ROOT:
        candidates.append(os.path.join(DRIVE_ROOT, "data", "humaneval_rs.jsonl"))
    candidates += ["data/humaneval_rs.jsonl", "../data/humaneval_rs.jsonl"]  # repo checkout
    for path in candidates:
        if os.path.exists(path):
            with open(path) as f:
                problems = [json.loads(line) for line in f if line.strip()]
            print(f"Loaded {len(problems)} problems from cache: {path}")
            return problems

    # Last resort (no Hub involved): hand-upload the repo's data/humaneval_rs.jsonl,
    # then stash it on Drive so this never happens again.
    if IN_COLAB:
        from google.colab import files
        print("Benchmark not found on Drive. Upload data/humaneval_rs.jsonl from the repo:")
        uploaded = files.upload()
        raw = next(iter(uploaded.values()))
        problems = [json.loads(line) for line in raw.decode().splitlines() if line.strip()]
        if DRIVE_ROOT:
            dest = os.path.join(DRIVE_ROOT, "data", "humaneval_rs.jsonl")
            with open(dest, "wb") as f:
                f.write(raw)
            print("Cached to Drive:", dest)
        return problems
    raise FileNotFoundError("humaneval_rs.jsonl not found — expected in the repo's data/ "
                            "folder or on Drive under CodeGen_Group45/data/.")

ds = load_benchmark()
assert len(ds) == 156, f"expected 156 problems, got {len(ds)}"
assert all(k in ds[0] for k in ("name", "prompt", "tests", "stop_tokens"))

# Look at one problem so the format is concrete
ex = ds[0]
print("\n===== PROMPT (given) =====\n", ex["prompt"])
print("===== TESTS (given) =====\n", ex["tests"])
print("===== stop tokens =====", ex["stop_tokens"])

Loaded 156 problems from cache: /content/drive/MyDrive/CodeGen_Group45/data/humaneval_rs.jsonl

===== PROMPT (given) =====
 /// Check if in given vector of numbers, are any two numbers closer to each other than
/// given threshold.
/// >>> has_close_elements(vec![1.0, 2.0, 3.0], 0.5)
/// false
/// >>> has_close_elements(vec![1.0, 2.8, 3.0, 4.0, 5.0, 2.0], 0.3)
/// true
fn has_close_elements(numbers: Vec<f64>, threshold: f64) -> bool {

===== TESTS (given) =====
 }

fn main() {
    let candidate = has_close_elements;
    assert_eq!(candidate(vec![1.0, 2.0, 3.9, 4.0, 5.0, 2.2], 0.3), true);
    assert_eq!(candidate(vec![1.0, 2.0, 3.9, 4.0, 5.0, 2.2], 0.05), false);
    assert_eq!(candidate(vec![1.0, 2.0, 5.9, 4.0, 5.0], 0.95), true);
    assert_eq!(candidate(vec![1.0, 2.0, 5.9, 4.0, 5.0], 0.8), false);
    assert_eq!(candidate(vec![1.0, 2.0, 3.0, 4.0, 5.0, 2.0], 0.1), true);
    assert_eq!(candidate(vec![1.1, 2.2, 3.1, 4.1, 5.1], 1.0), true);
    assert_eq!(candidate(vec![1.1, 2.2, 3.1, 

## 4. The harness function
This is the heart of Step 1. It glues the three parts into one `main.rs`, compiles it,
runs it, and returns one of: `pass`, `compile_error`, `run_fail`, `compile_timeout`, `run_timeout`.

In [5]:
import subprocess, tempfile, os

def evaluate_one(prompt, completion, tests, compile_timeout=60, run_timeout=10):
    """Assemble prompt+completion+tests into a Rust program, compile and run it."""
    program = prompt + completion + tests
    with tempfile.TemporaryDirectory() as wd:
        src  = os.path.join(wd, "main.rs")
        binp = os.path.join(wd, "prog")
        with open(src, "w") as f:
            f.write(program)

        # 1) compile
        try:
            c = subprocess.run(["rustc", src, "-o", binp],
                               capture_output=True, text=True, timeout=compile_timeout)
        except subprocess.TimeoutExpired:
            return "compile_timeout"
        if c.returncode != 0:
            return "compile_error"          # didn't even build

        # 2) run against the tests
        try:
            r = subprocess.run([binp], capture_output=True, text=True, timeout=run_timeout)
        except subprocess.TimeoutExpired:
            return "run_timeout"             # probably an infinite loop
        return "pass" if r.returncode == 0 else "run_fail"

print("harness ready")

harness ready


## 5. We self-test the harness (most important step)

---


Before we trust the harness, we prove it gives the right verdict on code we already know is
correct / wrong / broken. If these three checks don't come out as we expect, the bug is in our
**harness**, not in any model.

In [6]:
ex = ds[0]   # HumanEval_0: has_close_elements(numbers: Vec<f64>, threshold: f64) -> bool

# (a) a CORRECT body  -> should PASS
correct_body = """
    for i in 0..numbers.len() {
        for j in 0..numbers.len() {
            if i != j && (numbers[i] - numbers[j]).abs() < threshold {
                return true;
            }
        }
    }
    return false;
"""

# (b) a WRONG body (compiles, but fails the tests) -> should RUN_FAIL
wrong_body = "\n    return false;\n"

# (c) a BROKEN body (does not compile) -> should COMPILE_ERROR
broken_body = "\n    return this_is_not_defined;\n"

print("correct ->", evaluate_one(ex["prompt"], correct_body, ex["tests"]))
print("wrong   ->", evaluate_one(ex["prompt"], wrong_body,   ex["tests"]))
print("broken  ->", evaluate_one(ex["prompt"], broken_body,  ex["tests"]))

assert evaluate_one(ex["prompt"], correct_body, ex["tests"]) == "pass"
assert evaluate_one(ex["prompt"], wrong_body,   ex["tests"]) == "run_fail"
assert evaluate_one(ex["prompt"], broken_body,  ex["tests"]) == "compile_error"
print("\n Harness works correctly — it can tell good Rust from bad.")

correct -> pass
wrong   -> run_fail
broken  -> compile_error

 Harness works correctly — it can tell good Rust from bad.


## 6. The model — vanilla Qwen2.5-Coder-1.5B (fp16)

Same acquisition ladder as Step 5 (that notebook has the full story): **Drive copy**
(trusted only with the `_SAVED_OK` marker, copied to local disk before loading) →
**ModelScope** (primary hub — HF kept stalling from Colab in July 2026) → **HF Hub**
(last resort, inside a killable subprocess, because a stalled Xet download hangs
forever instead of raising). After Step 5's run the fp16 copy is already on Drive,
so this normally takes ~2 minutes and touches no hub at all.

In [7]:
# Do NOT add `torch` (Colab's preinstalled torch already matches its CUDA stack)
# and do NOT add `datasets` (see Section 2).
!pip install -q -U transformers accelerate
print("done")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 69.4 MB/s eta 0:00:00
done


In [8]:
import os, shutil, subprocess, sys

MODEL_ID  = "Qwen/Qwen2.5-Coder-1.5B"
MARKER    = "_SAVED_OK"   # written only after a COMPLETE save to Drive
DRIVE_MODEL_DIR = os.path.join(DRIVE_ROOT, "models", "qwen25coder-1p5b") if DRIVE_ROOT else None
LOCAL_DIR = "/content/qwen25coder-1p5b"

def _modelscope_download():
    # Alibaba's hub — Qwen's home turf, same files, zero HF infrastructure.
    # Verified 2026-07-14: modelscope 1.38's entire dep closure is
    # requests/tqdm/urllib3/packaging/filelock/modelscope-hub — no datasets, no
    # pyarrow — so a plain install cannot trigger the Colab pyarrow crash (Section 2).
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "modelscope"],
                   check=True)
    from modelscope import snapshot_download
    return snapshot_download(MODEL_ID)

def _hf_download():
    # HF-from-Colab stalls mid-download (Xet, July 2026), and a stall HANGS forever
    # instead of raising — so the download runs in a subprocess we can kill on timeout.
    # snapshot_download resumes partial downloads, so a killed attempt costs nothing.
    code = f"from huggingface_hub import snapshot_download; snapshot_download('{MODEL_ID}')"
    for attempt in (1, 2):
        try:
            subprocess.run([sys.executable, "-c", code], check=True, timeout=900)
            from huggingface_hub import snapshot_download
            return snapshot_download(MODEL_ID, local_files_only=True)  # already cached
        except subprocess.TimeoutExpired:
            print(f"HF Hub attempt {attempt}: no finish within 15 min (stalled) — killed")
        except subprocess.CalledProcessError:
            print(f"HF Hub attempt {attempt}: download process errored")
    raise RuntimeError(
        "All hubs failed (Drive empty, ModelScope failed, HF stalled/errored twice). "
        "Check the Colab proxy/network, or download the model on another machine and "
        "upload it to Drive under models/qwen25coder-1p5b with an empty _SAVED_OK file.")

def fetch_model_dir():
    """Return a local directory holding the model files. Order: Drive -> ModelScope -> HF Hub."""
    # (1) Drive copy. Copy to local disk first: reading 3 GB straight off the Drive
    #     FUSE mount is slow and occasionally errors out mid-load.
    if DRIVE_MODEL_DIR and os.path.exists(os.path.join(DRIVE_MODEL_DIR, MARKER)):
        if not os.path.exists(os.path.join(LOCAL_DIR, MARKER)):
            print("Model found on Drive — copying to local disk (one-time per session)...")
            shutil.copytree(DRIVE_MODEL_DIR, LOCAL_DIR, dirs_exist_ok=True)
        print("Using the Drive copy")
        return LOCAL_DIR

    # (2) ModelScope — primary hub while HF-from-Colab is broken.
    try:
        path = _modelscope_download()
        print("Downloaded from ModelScope")
        return path
    except Exception as e:
        print(f"ModelScope failed: {type(e).__name__}: {e}")

    # (3) HF Hub — last resort, stall-proofed.
    path = _hf_download()
    print("Downloaded from the Hugging Face Hub")
    return path

model_dir = fetch_model_dir()
print("model files at:", model_dir)

Model found on Drive — copying to local disk (one-time per session)...
Using the Drive copy
model files at: /content/qwen25coder-1p5b


In [9]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

assert torch.cuda.is_available(), "No GPU — Runtime -> Change runtime type -> T4 GPU, then rerun."

tok = AutoTokenizer.from_pretrained(model_dir)
model = AutoModelForCausalLM.from_pretrained(model_dir, dtype=torch.float16).to("cuda")  # T4 has no bf16
model.eval()
print("model loaded on", model.device)

# One-time: stash an fp16 copy on Drive so no future session ever needs a hub again.
if DRIVE_MODEL_DIR and not os.path.exists(os.path.join(DRIVE_MODEL_DIR, MARKER)):
    print("Saving fp16 copy to Drive (one-time, ~3 GB, takes a few minutes)...")
    model.save_pretrained(DRIVE_MODEL_DIR)
    tok.save_pretrained(DRIVE_MODEL_DIR)
    with open(os.path.join(DRIVE_MODEL_DIR, MARKER), "w") as f:
        f.write("ok\n")
    print("Saved to", DRIVE_MODEL_DIR)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

model loaded on cuda:0


In [10]:
def trim_to_body(text):
    # Cut at the brace that closes the function, IGNORING braces inside strings/chars/comments.
    depth = 1
    i, n = 0, len(text)
    in_str = in_char = in_line = in_block = False
    while i < n:
        ch = text[i]
        nxt = text[i+1] if i+1 < n else ""
        if in_line:
            if ch == "\n": in_line = False
            i += 1; continue
        if in_block:
            if ch == "*" and nxt == "/": in_block = False; i += 2; continue
            i += 1; continue
        if in_str:
            if ch == "\\": i += 2; continue
            if ch == '"': in_str = False
            i += 1; continue
        if in_char:
            if ch == "\\": i += 2; continue
            if ch == "'": in_char = False
            i += 1; continue
        if ch == "/" and nxt == "/": in_line = True; i += 2; continue
        if ch == "/" and nxt == "*": in_block = True; i += 2; continue
        if ch == '"': in_str = True; i += 1; continue
        if ch == "'":
            if nxt == "\\" or (i+2 < n and text[i+2] == "'"): in_char = True
            i += 1; continue
        if ch == "{": depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0: return text[:i]
        i += 1
    return text


def qwen_completion(ex, max_new_tokens=512):
    inputs = tok(ex["prompt"], return_tensors="pt").to(model.device)
    with torch.inference_mode():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens,
                             do_sample=False, pad_token_id=tok.eos_token_id,
                             stop_strings=["\n}"], tokenizer=tok)  # MultiPL-E's stop token — saves GPU time
    text = tok.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    return trim_to_body(text)

print("completion fn ready")

completion fn ready


## 7. The RAG corpus — our MBPP translation pairs

The retrieval corpus is the same one Step 4 used at 350M: our validated MBPP
Python→Rust pairs. Each `rust_solution` is a complete, compiling function with its
`///` doc comment — exactly the shape of what we ask the model to write. As in
Step 4 we keep one exemplar per task (the shortest passing solution) to save
context tokens, and index with TF-IDF over character 3–5-grams.

One deliberate change from Step 4: **the query is the MultiPL-E prompt itself**
(doc comment + signature), and the document side is each pair's doc comment +
signature to match. Step 4 retrieved by Python-source similarity because its model
was a Python→Rust translator with the Python in hand; the vanilla completion task
has no Python at inference time, and retrieval must live with the same information
the model gets.

Source order: Drive `pairs_v3.jsonl` → `pairs_v2.jsonl` → the repo's
`notebooks/pairs.jsonl` (v1, 194 pairs — smaller, but keeps the notebook runnable
with zero Drive access). scikit-learn is preinstalled on Colab.

In [11]:
import json, os
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def load_pairs():
    candidates = []
    if DRIVE_ROOT:
        candidates += [os.path.join(DRIVE_ROOT, c) for c in ("pairs_v3.jsonl", "pairs_v2.jsonl")]
    candidates += ["notebooks/pairs.jsonl", "pairs.jsonl", "../notebooks/pairs.jsonl"]
    for path in candidates:
        if os.path.exists(path):
            with open(path) as f:
                loaded = [json.loads(l) for l in f if l.strip()]
            print(f"Loaded {len(loaded)} pairs from {path}")
            if "pairs_v" not in os.path.basename(path):
                print("WARNING: this is the v1 fallback corpus (194 pairs). Fine for a dry run; "
                      "the real sweep should see pairs_v3 on Drive (643 pairs).")
            return loaded
    raise FileNotFoundError("no pairs file found (Drive pairs_v3/v2 or repo notebooks/pairs.jsonl)")

pairs = load_pairs()

# One exemplar per task: the SHORTEST solution (saves context tokens) — as in Step 4.
best = {}
for p in pairs:
    if p["task"] not in best or len(p["rust_solution"]) < len(best[p["task"]]["rust_solution"]):
        best[p["task"]] = p
corpus = list(best.values())

def doc_text(p):
    # What the retriever "sees" for an exemplar: doc comment + signature only,
    # to match the query (the MultiPL-E prompt = doc comment + signature).
    return p["rust_prompt"]

vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), max_features=50000)
X = vec.fit_transform([doc_text(p) for p in corpus])
print(len(corpus), "unique tasks in the RAG index (from", len(pairs), "pairs)")

# Sanity check: an exemplar's own prompt must retrieve that exemplar first.
sims = cosine_similarity(vec.transform([doc_text(corpus[0])]), X)[0]
assert corpus[int(sims.argmax())]["task"] == corpus[0]["task"], "retriever failed its self-check"
print("retriever self-check OK")

Loaded 643 pairs from /content/drive/MyDrive/CodeGen_Group45/pairs_v3.jsonl
251 unique tasks in the RAG index (from 643 pairs)
retriever self-check OK


### 7b. The idiom exemplars — an error-analysis-guided corpus row

Step 5's error analysis says the compile errors concentrate in a few idioms:
indexing with `isize`, ordering floats, int/float mixing, helper functions that
were never defined, values moved instead of borrowed. The ten exemplars below
demonstrate exactly those idioms. They were written from the rustc error
*buckets*, not from any benchmark problem or its solution, so the corpus stays
disjoint from HumanEval.

**Design decision (measured, not guessed):** we first tried simply merging these
ten into the MBPP index — but TF-IDF then surfaces an idiom exemplar in the top-2
for only 4 of the 34 compile-error problems (the MBPP corpus is 20–60× larger and
usually closer in wording). Relevance-gated inclusion would make the row a no-op.
So the idiom row instead **always prepends the single most similar idiom
exemplar** (retrieved from the ten by the same TF-IDF machinery) on top of the K
retrieved MBPP exemplars: one targeted Rust lesson per prompt, guaranteed present.

**Honest caveat for the report:** the bucket frequencies were measured on the eval
set itself, so this row is development-set-informed in a way the plain MBPP sweep
is not. We keep both rows and say so.

In [12]:
IDIOM_EXEMPLARS = [
    {"task": "idiom_index_with_cast",
     "rust_prompt": "/// Return the element at position i of a list, where i is given as an isize.\nfn element_at(values: Vec<isize>, i: isize) -> isize {",
     "rust_solution": "/// Return the element at position i of a list, where i is given as an isize.\nfn element_at(values: Vec<isize>, i: isize) -> isize {\n    values[i as usize]\n}"},
    {"task": "idiom_max_of_floats",
     "rust_prompt": "/// Return the largest value in a non-empty list of floats.\nfn max_float(values: Vec<f64>) -> f64 {",
     "rust_solution": "/// Return the largest value in a non-empty list of floats.\nfn max_float(values: Vec<f64>) -> f64 {\n    values.iter().cloned().fold(f64::NEG_INFINITY, f64::max)\n}"},
    {"task": "idiom_sort_floats",
     "rust_prompt": "/// Return a copy of the list of floats sorted in increasing order.\nfn sort_floats(values: Vec<f64>) -> Vec<f64> {",
     "rust_solution": "/// Return a copy of the list of floats sorted in increasing order.\nfn sort_floats(values: Vec<f64>) -> Vec<f64> {\n    let mut sorted = values.clone();\n    sorted.sort_by(|a, b| a.partial_cmp(b).unwrap());\n    sorted\n}"},
    {"task": "idiom_nested_helper_fn",
     "rust_prompt": "/// Return true if the sum of the digits of n is a prime number.\nfn digit_sum_is_prime(n: isize) -> bool {",
     "rust_solution": "/// Return true if the sum of the digits of n is a prime number.\nfn digit_sum_is_prime(n: isize) -> bool {\n    fn is_prime(x: isize) -> bool {\n        if x < 2 {\n            return false;\n        }\n        for d in 2..=((x as f64).sqrt() as isize) {\n            if x % d == 0 {\n                return false;\n            }\n        }\n        true\n    }\n    let mut s = 0;\n    let mut m = n.abs();\n    while m > 0 {\n        s += m % 10;\n        m /= 10;\n    }\n    is_prime(s)\n}"},
    {"task": "idiom_int_float_mix",
     "rust_prompt": "/// Return the average of a non-empty list of integers as a float.\nfn average(values: Vec<isize>) -> f64 {",
     "rust_solution": "/// Return the average of a non-empty list of integers as a float.\nfn average(values: Vec<isize>) -> f64 {\n    let total: isize = values.iter().sum();\n    total as f64 / values.len() as f64\n}"},
    {"task": "idiom_widen_to_avoid_overflow",
     "rust_prompt": "/// Return the product of all elements, computed in i64 so it cannot overflow i32.\nfn product_wide(values: Vec<i32>) -> i64 {",
     "rust_solution": "/// Return the product of all elements, computed in i64 so it cannot overflow i32.\nfn product_wide(values: Vec<i32>) -> i64 {\n    values.iter().map(|&v| v as i64).product()\n}"},
    {"task": "idiom_string_compare",
     "rust_prompt": "/// Return true if two words are equal ignoring case.\nfn same_word(a: String, b: String) -> bool {",
     "rust_solution": "/// Return true if two words are equal ignoring case.\nfn same_word(a: String, b: String) -> bool {\n    a.to_lowercase() == b.to_lowercase()\n}"},
    {"task": "idiom_char_at_position",
     "rust_prompt": "/// Return the character at position k of a string (character, not byte, index).\nfn char_at(s: String, k: usize) -> char {",
     "rust_solution": "/// Return the character at position k of a string (character, not byte, index).\nfn char_at(s: String, k: usize) -> char {\n    s.chars().nth(k).unwrap_or(' ')\n}"},
    {"task": "idiom_iterate_by_reference",
     "rust_prompt": "/// Return the difference between the largest and smallest element,\nfn value_range(values: Vec<isize>) -> isize {",
     "rust_solution": "/// Return the difference between the largest and smallest element,\n/// iterating by reference so the vector is not moved.\nfn value_range(values: Vec<isize>) -> isize {\n    let max = values.iter().max().unwrap();\n    let min = values.iter().min().unwrap();\n    max - min\n}"},
    {"task": "idiom_enumerate_positions",
     "rust_prompt": "/// Return the positions at which a negative number appears in the list.\nfn negative_positions(values: Vec<isize>) -> Vec<usize> {",
     "rust_solution": "/// Return the positions at which a negative number appears in the list.\nfn negative_positions(values: Vec<isize>) -> Vec<usize> {\n    values.iter().enumerate()\n          .filter(|&(_, &v)| v < 0)\n          .map(|(i, _)| i)\n          .collect()\n}"},
]

# A separate tiny index over just the ten idioms: the idiom row always shows the
# ONE most similar idiom exemplar (see the markdown above for why it is not
# simply merged into the MBPP index).
vec_id = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), max_features=50000)
X_id = vec_id.fit_transform([doc_text(p) for p in IDIOM_EXEMPLARS])

def retrieve(query_text, k):
    if k <= 0:
        return []
    sims = cosine_similarity(vec.transform([query_text]), X)[0]
    return [corpus[i] for i in sims.argsort()[::-1][:k]]

def retrieve_idiom(query_text):
    sims = cosine_similarity(vec_id.transform([query_text]), X_id)[0]
    return IDIOM_EXEMPLARS[int(sims.argmax())]

demo_q = "/// Return the largest element of a list of floats.\nfn max_elem(values: Vec<f64>) -> f64 {\n"
print("nearest idiom for a float task:", retrieve_idiom(demo_q)["task"])
print("nearest MBPP exemplars:        ", [d["task"] for d in retrieve(demo_q, 2)])

nearest idiom for a float task: idiom_max_of_floats
nearest MBPP exemplars:         ['mbpp_618_div_list', 'mbpp_251_insert_element']


## 8. RAG prompt assembly

Exemplars go ABOVE the problem prompt, as complete functions separated by blank
lines — pure Rust, the same completion format the baseline used. Nothing is added
to the problem prompt itself, so **K=0 is byte-identical to Step 5's prompt** —
asserted below for all 156 problems. That, plus greedy decoding, is what lets the
K=0 control reproduce 37.8% exactly.

A token budget caps the prompt: Qwen's 32K context is not the constraint, but long
prompts are slow on a T4. Same policy as Step 4 — nearest exemplar first, and an
exemplar that would overflow the budget is skipped, not truncated.

In [13]:
PROMPT_BUDGET = 4096 - 512   # prompt tokens; plenty for K=4 (median exemplar is small)

def ntokens(s):
    return len(tok(s)["input_ids"])

def build_rag_prompt(ex, k, use_idioms=False):
    exemplars = retrieve(ex["prompt"], k)                         # nearest first
    if use_idioms:
        exemplars = [retrieve_idiom(ex["prompt"])] + exemplars    # the one idiom lesson on top
    blocks, total = [], ntokens(ex["prompt"])
    for exemplar in exemplars:
        block = exemplar["rust_solution"].rstrip() + "\n\n"
        bt = ntokens(block)
        if total + bt > PROMPT_BUDGET:
            continue                     # would overflow the budget — skip, try next-nearest
        blocks.append(block)
        total += bt
    return "".join(blocks) + ex["prompt"], len(blocks)

# K=0 must be the baseline prompt, byte for byte — on every problem.
assert all(build_rag_prompt(ex, 0)[0] == ex["prompt"] for ex in ds)
print("K=0 identity check passed on all", len(ds), "problems")

demo_prompt, n_used = build_rag_prompt(ds[2], 2)
print(f"\ndemo: K=2 used {n_used} exemplars, {ntokens(demo_prompt)} prompt tokens")
print("-" * 60)
print(demo_prompt[:1200])

K=0 identity check passed on all 156 problems

demo: K=2 used 2 exemplars, 233 prompt tokens
------------------------------------------------------------
/// Write a function to convert the given decimal number to its binary equivalent, represented as a string with no leading zeros.
fn decimal_to_binary(n: isize) -> String {
    let mut binary_str = String::new();
    let mut n = n;
    loop {
        let rem = n % 2;
        binary_str.push_str(&rem.to_string());
        n /= 2;
        if n == 0 {
            break;
        }
    }
    binary_str.chars().rev().collect()
}

/// Write a rsthon function to find smallest number in a vector.
fn smallest_num(xs: Vec<isize>) -> isize {
    xs.iter().fold(isize::MAX, |a, &b| a.min(b))
}

/// Given a positive floating point number, it can be decomposed into
/// and integer part (largest integer smaller than given number) and decimals
/// (leftover part always smaller than 1).
/// Return the decimal part of the number.
/// >>> truncate_number(

## 9. The crash-safe sweep runner (+ smoke test)

Same contract as every GPU run in this project: each configuration streams
per-problem results to its own jsonl on Drive, resumes where it left off, and
skips finished problems. Generation parameters are IDENTICAL to Step 5 (greedy,
`max_new_tokens=512`, stop at `"\n}"`), so K=0 is a true rerun of the baseline.

Smoke test first (house rule): 5 problems at K=2, throwaway file, and we refuse to
continue if every one of them comes back `compile_error`.

In [14]:
import time
from collections import Counter

EVAL_DIR = os.path.join(DRIVE_ROOT, "eval") if DRIVE_ROOT else "."

def rag_completion(prompt_text, max_new_tokens=512):
    inputs = tok(prompt_text, return_tensors="pt").to(model.device)
    with torch.inference_mode():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens,
                             do_sample=False, pad_token_id=tok.eos_token_id,
                             stop_strings=["\n}"], tokenizer=tok)  # MultiPL-E's stop token
    text = tok.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    return trim_to_body(text)

def run_k(k, use_idioms=False, limit=None, tag=""):
    path = os.path.join(EVAL_DIR, f"step6_qwen_rag_k{k}{tag}.jsonl")
    done = {}
    if os.path.exists(path):
        with open(path) as f:
            for line in f:
                rec = json.loads(line)
                done[rec["name"]] = rec["status"]
    data = ds if limit is None else ds[:limit]
    todo = [ex for ex in data if ex["name"] not in done]
    print(f"K={k}{tag}: {len(done)} already done, {len(todo)} to go -> {path}")
    t0 = time.time()
    with open(path, "a") as out:
        for ex in todo:
            prompt_text, n_used = build_rag_prompt(ex, k, use_idioms)
            body = rag_completion(prompt_text)
            status = evaluate_one(ex["prompt"], body, ex["tests"])
            out.write(json.dumps({"name": ex["name"], "k": k, "n_examples": n_used,
                                  "status": status, "body": body}) + "\n")
            out.flush()
            done[ex["name"]] = status
            print(f"[{len(done):3d}/{len(data)}] {ex['name'][:40]:40s} {status:14s} ({time.time()-t0:5.0f}s)")
    counts = Counter(done.values())
    print(f"K={k}{tag}: pass {100*counts['pass']/len(done):.1f}%  {dict(counts)}")
    return counts

# --- smoke test: 5 problems at K=2, throwaway file (house rule) ---
smoke = run_k(2, limit=5, tag="_smoke")
os.remove(os.path.join(EVAL_DIR, "step6_qwen_rag_k2_smoke.jsonl"))
assert smoke["compile_error"] < 5, ("every smoke problem failed to compile — print one "
                                    "assembled prompt and body before spending GPU time")
print("\nsmoke OK — the full sweep below is safe to launch")

K=2_smoke: 0 already done, 5 to go -> /content/drive/MyDrive/CodeGen_Group45/eval/step6_qwen_rag_k2_smoke.jsonl
[  1/5] HumanEval_0_has_close_elements           pass           (    8s)
[  2/5] HumanEval_1_separate_paren_groups        run_fail       (   14s)
[  3/5] HumanEval_2_truncate_number              pass           (   16s)
[  4/5] HumanEval_3_below_zero                   pass           (   19s)
[  5/5] HumanEval_4_mean_absolute_deviation      compile_error  (   23s)
K=2_smoke: pass 60.0%  {'pass': 3, 'run_fail': 1, 'compile_error': 1}

smoke OK — the full sweep below is safe to launch


## 10. The full sweep — K = 0 / 1 / 2 / 4

**Sharding across Colab accounts:** each K writes its own Drive file, so different
accounts can safely take different K values — edit `RUN_KS` per account (e.g.
account A runs `[0, 1]`, account B `[2, 4]`). Never point two live sessions at the
SAME K: they would append to one file concurrently.

**The control:** K=0 must land on 37.8% (59/156). If it does not, something
drifted from Step 5 — stop and diff the pipeline before trusting any K>0 row.

Every run resumes from its Drive file, so a disconnect costs nothing but the
problem that was in flight.

In [15]:
RUN_KS = [0, 1, 2, 4]        # edit per account when sharding
for k in RUN_KS:
    run_k(k)

K=0: 0 already done, 156 to go -> /content/drive/MyDrive/CodeGen_Group45/eval/step6_qwen_rag_k0.jsonl
[  1/156] HumanEval_0_has_close_elements           pass           (    3s)
[  2/156] HumanEval_1_separate_paren_groups        run_fail       (    8s)
[  3/156] HumanEval_2_truncate_number              pass           (   10s)
[  4/156] HumanEval_3_below_zero                   pass           (   14s)
[  5/156] HumanEval_4_mean_absolute_deviation      compile_error  (   17s)
[  6/156] HumanEval_5_intersperse                  pass           (   20s)
[  7/156] HumanEval_6_parse_nested_parens          run_fail       (   25s)
[  8/156] HumanEval_7_filter_by_substring          pass           (   28s)
[  9/156] HumanEval_8_sum_product                  pass           (   30s)
[ 10/156] HumanEval_9_rolling_max                  run_fail       (   33s)
[ 11/156] HumanEval_10_make_palindrome             compile_error  (   38s)
[ 12/156] HumanEval_11_string_xor                  pass           (   42s

### 10b. The idiom row

Same runner; the prompt gets the single most similar idiom exemplar on top of the
K=2 MBPP exemplars (see §7b for why it is guaranteed rather than relevance-gated).
We pin K=2 — the middle of the sweep; edit the call if the sweep clearly favors
another K. This is the row the error analysis predicts should move *compile*
errors specifically, so in the results, watch the `compile_err` column, not just
`pass%`.

In [16]:
run_k(2, use_idioms=True, tag="_idiom")

K=2_idiom: 0 already done, 156 to go -> /content/drive/MyDrive/CodeGen_Group45/eval/step6_qwen_rag_k2_idiom.jsonl
[  1/156] HumanEval_0_has_close_elements           run_fail       (    2s)
[  2/156] HumanEval_1_separate_paren_groups        run_fail       (    8s)
[  3/156] HumanEval_2_truncate_number              pass           (   10s)
[  4/156] HumanEval_3_below_zero                   pass           (   12s)
[  5/156] HumanEval_4_mean_absolute_deviation      compile_error  (   15s)
[  6/156] HumanEval_5_intersperse                  compile_error  (   18s)
[  7/156] HumanEval_6_parse_nested_parens          run_fail       (   22s)
[  8/156] HumanEval_7_filter_by_substring          pass           (   25s)
[  9/156] HumanEval_8_sum_product                  pass           (   27s)
[ 10/156] HumanEval_9_rolling_max                  run_fail       (   30s)
[ 11/156] HumanEval_10_make_palindrome             run_fail       (   36s)
[ 12/156] HumanEval_11_string_xor                  pass      

Counter({'run_fail': 65, 'pass': 51, 'compile_error': 39, 'run_timeout': 1})

## 11. Results — the RAG ablation at 1.5B

The table reads straight from the Drive files, so it can be re-run any time,
including while other accounts are still filling in their K values (incomplete
rows are flagged).

In [17]:
import glob

print("Reference: Qwen2.5-Coder-1.5B vanilla (Step 5) = 37.8% (59/156)")
print("           350M sweep (Step 4): K=0 7.1% | K=1 5.8% | K=2 4.5% | K=4 5.8% | cascade 10.3%")
print()
rows = []
for path in sorted(glob.glob(os.path.join(EVAL_DIR, "step6_qwen_rag_k*.jsonl"))):
    if "_smoke" in path:
        continue
    with open(path) as f:
        recs = [json.loads(l) for l in f]
    if not recs:
        continue
    label = os.path.basename(path).replace("step6_qwen_rag_", "").replace(".jsonl", "")
    c = Counter(r["status"] for r in recs)
    avg_ex = sum(r.get("n_examples", 0) for r in recs) / len(recs)
    rows.append((label, len(recs), 100 * c["pass"] / len(recs), c["compile_error"],
                 c["run_fail"], c["run_timeout"] + c["compile_timeout"], avg_ex))
print(f"{'config':>10} {'n':>4} {'pass%':>7} {'compile_err':>12} {'run_fail':>9} {'timeout':>8} {'avg examples':>13}")
for label, n, p, ce, rf, to, ae in rows:
    flag = "  (INCOMPLETE)" if n < len(ds) else ""
    print(f"{label:>10} {n:>4} {p:>6.1f}% {ce:>12} {rf:>9} {to:>8} {ae:>13.2f}{flag}")

Reference: Qwen2.5-Coder-1.5B vanilla (Step 5) = 37.8% (59/156)
           350M sweep (Step 4): K=0 7.1% | K=1 5.8% | K=2 4.5% | K=4 5.8% | cascade 10.3%

    config    n   pass%  compile_err  run_fail  timeout  avg examples
        k0  156   37.8%           34        62        1          0.00
        k1  156   34.6%           39        63        0          1.00
        k2  156   36.5%           36        63        0          2.00
  k2_idiom  156   32.7%           39        65        1          3.00
        k4  156   39.7%           33        60        1          4.00


## 12. Flip analysis + the compile-guided cascade

Two questions:

1. **Are the flips uniform?** If each K solves DIFFERENT problems (as at 350M,
   where 8 problems passed only with RAG), combining configurations is worth
   something even when no single K wins.
2. **The cascade:** answer at K=0; **only if that answer fails to compile**, fall
   back to the K>0 answer. The compiler's verdict is available at inference time,
   so this is a legal single-pipeline system, not an oracle. It costs no new GPU
   time — pure post-processing of the sweep files — and the error analysis gives
   it a hard target: the baseline's 34 compile errors (21.8 points of bounded
   headroom).

The oracle union is reported as an upper bound only — it peeks at test results and
is NOT a legal system.

In [18]:
def load_statuses(label):
    path = os.path.join(EVAL_DIR, f"step6_qwen_rag_{label}.jsonl")
    statuses = {}
    if os.path.exists(path):
        with open(path) as f:
            for line in f:
                rec = json.loads(line)
                statuses[rec["name"]] = rec["status"]
    return statuses

k0 = load_statuses("k0")
assert len(k0) == len(ds), "run K=0 first — the cascade starts from it"
variants = {lbl: load_statuses(lbl) for lbl in ("k1", "k2", "k4", "k2_idiom")}
variants = {lbl: st for lbl, st in variants.items() if len(st) == len(ds)}   # complete runs only

base_pass = {n for n, s in k0.items() if s == "pass"}
print(f"K=0 passes {len(base_pass)}")
for lbl, st in variants.items():
    vp = {n for n, s in st.items() if s == "pass"}
    print(f"{lbl:9s} passes {len(vp):3d} | only-{lbl} {len(vp - base_pass):2d} | lost vs K=0 {len(base_pass - vp):2d}")

union = set(base_pass)
for st in variants.values():
    union |= {n for n, s in st.items() if s == "pass"}
print(f"\noracle union (upper bound, NOT a system): {len(union)}/{len(ds)} = {100*len(union)/len(ds):.1f}%")

def cascade(order):
    passes = 0
    for name, status in k0.items():
        for lbl in order:
            if status != "compile_error":
                break
            status = variants[lbl][name]
        passes += (status == "pass")
    return passes

print("\ncompile-guided cascade (K=0 first, fall back only on compile_error):")
for order in ([l] for l in variants):
    p = cascade(order)
    print(f"  K=0 -> {' -> '.join(order):20s} {p}/{len(ds)} = {100*p/len(ds):.1f}%")
for order in (["k1", "k4"], ["k1", "k2", "k4"], ["k2_idiom", "k4"]):
    if all(lbl in variants for lbl in order):
        p = cascade(order)
        print(f"  K=0 -> {' -> '.join(order):20s} {p}/{len(ds)} = {100*p/len(ds):.1f}%")

K=0 passes 59
k1        passes  54 | only-k1 12 | lost vs K=0 17
k2        passes  57 | only-k2  9 | lost vs K=0 11
k4        passes  62 | only-k4 14 | lost vs K=0 11
k2_idiom  passes  51 | only-k2_idiom 14 | lost vs K=0 22

oracle union (upper bound, NOT a system): 85/156 = 54.5%

compile-guided cascade (K=0 first, fall back only on compile_error):
  K=0 -> k1                   64/156 = 41.0%
  K=0 -> k2                   66/156 = 42.3%
  K=0 -> k4                   67/156 = 42.9%
  K=0 -> k2_idiom             68/156 = 43.6%
  K=0 -> k1 -> k4             69/156 = 44.2%
  K=0 -> k1 -> k2 -> k4       69/156 = 44.2%
  K=0 -> k2_idiom -> k4       70/156 = 44.9%


## What this step adds

- The **RAG ablation at 1.5B** next to the identical ablation at 350M — the
  cross-scale story the report needs: does in-context help once the model is big
  enough to read it?
- An **error-analysis-guided corpus row** — the cheapest possible test of "teach
  the failing idioms in context", with its development-set-informed caveat stated.
- The **compile-guided cascade at 1.5B** — bounded in advance by the baseline's 34
  compile errors, so we know the maximum it can add before spending a minute of
  GPU time.

Next (per the handoff): the Step 3b translation LoRA on Qwen (risk: 600 MBPP pairs
can drag a strong 37.8% base down — measure, don't assume), then Qwen2.5-Coder-7B
in 4-bit as the "large LLM" row, then the CP3 comparison table.